# DataCo Supply Chain: Late-Delivery Analysis (pandas)

**Business question:** How much of DataCo's order volume ships late, and what is the financial value exposed to that risk?

**Author:** Charles Agyekum  
**Method:** Python / pandas. Rebuilds the DataCo Power BI analysis in code, with a data-quality gate before any number is trusted.

---

In [2]:
import pandas as pd

df = pd.read_csv("../DataCo_LateDelivery_PowerBI/DataCoSupplyChainDataSet.csv",encoding="latin-1")

df.shape

(180519, 53)

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 53 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Type                           180519 non-null  str    
 1   Days for shipping (real)       180519 non-null  int64  
 2   Days for shipment (scheduled)  180519 non-null  int64  
 3   Benefit per order              180519 non-null  float64
 4   Sales per customer             180519 non-null  float64
 5   Delivery Status                180519 non-null  str    
 6   Late_delivery_risk             180519 non-null  int64  
 7   Category Id                    180519 non-null  int64  
 8   Category Name                  180519 non-null  str    
 9   Customer City                  180519 non-null  str    
 10  Customer Country               180519 non-null  str    
 11  Customer Email                 180519 non-null  str    
 12  Customer Fname                 180519 non

In [4]:
df["Delivery Status"].value_counts()

Delivery Status
Late delivery        98977
Advance shipping     41592
Shipping on time     32196
Shipping canceled     7754
Name: count, dtype: int64

In [5]:
(df["Days for shipping (real)"] > df["Days for shipment (scheduled)"]).sum()

np.int64(103400)

In [6]:
df["Late_delivery_risk"].sum()

np.int64(98977)

In [7]:
df["derived_late"] = df["Days for shipping (real)"] > df["Days for shipment (scheduled)"]

df.loc[df["derived_late"], "Delivery Status"].value_counts()

Delivery Status
Late delivery        98977
Shipping canceled     4423
Name: count, dtype: int64

In [8]:
df.loc[df["derived_late"] & (df["Delivery Status"] == "Shipping canceled"), ["Days for shipping (real)", "Days for shipment (scheduled)"]].describe()

,Days for shipping (real),Days for shipment (scheduled)
count,4423.000000,4423.000000
mean,4.025096,2.440651
std,1.742738,1.391823
min,1.000000,0.000000
25%,2.000000,1.000000
50%,5.000000,2.000000
75%,6.000000,4.000000
max,6.000000,4.000000


## Defining "late delivery" - decision and evidence

Three candidate definitions were tested:
- 'Delivery Status =="Late delivery" '--> 98,977
- 'Late_delivery_risk == 1' --> 98,977
- Derived ('real days > scheduled days') --> 103,400

The derived rule counts 4,423 more orders. Cross-tab shows these are all 'Shipping canceled'.
Checked their shipping days: mean real = 4.0 vs scheduled 2.4 (min 1), so they *did* ship and were behind schedule when canceled - not warehouse cancellations.

**Decision:** late-delivery rate = **98,977 / 180,519 = 54.8%**, excluding canceled orders (never delivered to customer, so not a late *delivery*).

**Flagged separately:** 4,423 orders shipped late *and* were then canceled - a compound service-and-revenue failure worth its own attention.
    

In [10]:
late = df["Delivery Status"] == "Late delivery"

print("Late-delivered orders:", late.sum())
print("Revenue (Sales) on late orders:", df.loc[late, "Sales"].sum())
print("Profit (Benefit per order) on late orders:", df.loc[late, "Benefit per order"].sum())

Late-delivered orders: 98977
Revenue (Sales) on late orders: 20126395.26523882
Profit (Benefit per order) on late orders: 2140051.6816967707


In [12]:
df.groupby("Shipping Mode")["Late_delivery_risk"].mean().sort_values(ascending=False)

Shipping Mode
First Class       0.953225
Second Class      0.766328
Same Day          0.457430
Standard Class    0.380717
Name: Late_delivery_risk, dtype: float64

In [13]:
df.groupby("Shipping Mode")["Late_delivery_risk"].agg(["mean", "count"]).sort_values("mean", ascending=False)

,mean,count
Shipping Mode,,
First Class,0.953225,27814
Second Class,0.766328,35216
Same Day,0.457430,9737
Standard Class,0.380717,107752


In [14]:
df.groupby("Shipping Mode")[["Days for shipment (scheduled)", "Days for shipping (real)"]].mean()

,Days for shipment (scheduled),Days for shipping (real)
Shipping Mode,,
First Class,1.0,2.000000
Same Day,0.0,0.478279
Second Class,2.0,3.990828
Standard Class,4.0,3.995907


In [15]:
std = df[df["Shipping Mode"] == "Standard Class"]

std.groupby("Market")["Late_delivery_risk"].agg(["mean","count"]).sort_values("mean", ascending=False)

,mean,count
Market,,
Pacific Asia,0.384731,24586
Africa,0.382140,7055
Europe,0.380094,29740
LATAM,0.378868,31119
USCA,0.378573,15252


In [19]:
std = df[df["Shipping Mode"] == "Standard Class"].copy()
std["order_year"] = pd.to_datetime(std["order date (DateOrders)"]).dt.year

std.groupby("order_year")["Late_delivery_risk"].agg(["mean", "count"])

,mean,count
order_year,,
2015,0.379439,37421
2016,0.384001,37177
2017,0.377991,31924
2018,0.391057,1230


In [20]:
std["Days for shipping (real)"].describe()

count    107752.000000
mean          3.995907
std           1.417266
min           2.000000
25%           3.000000
50%           4.000000
75%           5.000000
max           6.000000
Name: Days for shipping (real), dtype: float64

### Recommendations

**Diagnosis.** The 54.8% late rate is a promise problem. The shipping SLAs sit at or below what the operation actually delivers, so orders miss targets they were never able to meet.

- First Class promises one day and takes two. It is genuinely fast, yet it breaks its own one-day promise 95% of the time.
- Standard Class promises four days and averages four. Normal variation of about 1.4 days then tips 38% of its orders over the line, roughly 41,000 orders, the largest single group.
- The rate stays near 38% across every region and year, which tells me the cause is systemic.

**Recommendation 1: recalibrate the shipping promises.** Reset each SLA to a level the operation reliably hits, around the 90th percentile of real delivery time. First Class moves from one day to two, Standard from four to five. Tens of thousands of orders fall back inside their promise at almost no cost, and customers get a promise the business actually keeps.

**Recommendation 2: reduce delivery-time variance.** Tighten the spread on Standard Class through more consistent fulfilment, so fewer orders drift into the late tail. This needs real operational investment and pays back over time. I would start with Standard Class for its volume and look hard at the third-party versus in-house shipping legs.

**Sequence.** Lead with the recalibration for a fast, credible win. Fund the variance work for the lasting fix.

**Money at stake.** \$2.14M of profit and \$20.1M of revenue sit on late-delivered orders.

**One more flag.** 4,423 orders shipped behind schedule and were then cancelled. That is a double loss of service and revenue, and it deserves its own review.


## Executive summary

I measured how much of DataCo's order volume ships late and what it costs.

- 54.8% of the 180,519 orders arrived late, which is 98,977 orders. I confirmed that figure from the raw shipping dates myself instead of relying on the vendor's status label.
- The late orders carry \$2.14M of profit and \$20.1M of revenue.
- First Class, the premium tier, misses its promise 95% of the time even though it ships in two days. Its one-day SLA is the real issue.
- Standard Class carries the most late orders in raw numbers, around 41,000, because it handles the most volume.
- The late rate holds near 38% in every region and every year, so the cause sits in the process itself.
- My recommendation: reset the shipping promises to levels the operation can actually hit, then work on how much delivery times vary.
